<a href="https://colab.research.google.com/github/DrGPCR/NEUR201/blob/main/Unit_3/notebooks/W11L2_Inferential_statistics__STUDENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W11 L2 — From Standard Errors to t-Tests: Is 4-AP's Effect Real?

**NEUR 201 — Research Methods & Data Analysis for Cellular Neuroscience**  ·  *In-class activity*

**Name:** *(double-click to type)*  **Date:** *(double-click to type)*

---

## Background

In W10 L1 you washed **4-AP** onto a brain slice and watched the event rate jump while the CONTROL
slice barely moved. It would be easy to stop there and call it an effect.

But one slice is not an experiment. Image a different slice and you'd get a different number — not
because the biology changed, but because every slice is a little different. A sceptic can always
ask: *maybe that DRUG slice was just a naturally active one?*

So the experiment was repeated: **10 slices per group**, each reduced to one number, and the question
becomes harder than "is DRUG bigger than CONTROL?":

> **Is the gap between the two groups bigger than the wobble you'd expect by chance?**

You will answer it with the **independent-samples t-test** from today's slides, one step at a time:
the **standard error** of each sample, the **t-score**, the **degrees of freedom**, and the
**p-value**, and then you will use the **null and alternative hypotheses** and **α** to decide.

**Why independent samples?** A slice is either CONTROL or DRUG, never both. The two samples are
mutually exclusive, so we use the independent-samples t-test (not the dependent/paired version, which
needs the *same* subjects measured twice).

## The data

For each slice we already computed its **mean ΔF/F₀ (%) during the application period** — one number
summarising how active that slice became after wash-on.

| Column | What it is |
|---|---|
| `Group` | `CONTROL` (aCSF vehicle) or `DRUG` (4-AP) |
| `Slice` | which slice this row came from |
| `dFF_percent` | that slice's mean ΔF/F₀ during application, in % |

**Why one value per slice, and not per cell?** The 100 cells in one recording are not 100 independent
experiments — they share a slice, a drug wash, a dissection and a day. Treating them as n = 100 is
**pseudoreplication**, and it makes every result look far more certain than it is. Each **slice** is
one independent data point, so **n is the number of slices**.

### Symbols used in this notebook

| Symbol | Read it as |
|---|---|
| $\bar{X}$, $s$ | the **sample** mean and standard deviation |
| $\mu$ | the **population** mean |
| $n$ | the number of scores (slices) in a group |
| $SE$ | the standard error of the mean (SEM) |
| $t$, $df$ | the t-score and its degrees of freedom |
| $H_0$, $H_a$ | the null and alternative hypotheses |
| $p$ | the p-value |
| $\alpha$ | the alpha level — the cut-off, usually 0.05 |

## What you'll be able to do by the end

1. Describe the difference between two group means you will have to judge
2. Calculate the **standard error** of each sample
3. Combine them into the **standard error of the difference** and compute the **t-score**
4. Calculate the **degrees of freedom** and the **p-value**, and show them on the t-distribution
5. State the **two-tailed null and alternative hypotheses**, compare **p** with **α = 0.05**, and decide
   whether the difference is **statistically significant**

## How this notebook works

**Run every cell in order, from the top.** Click a cell and press **Shift + Enter**.

| | What to do |
|---|---|
| **Ordinary cells** | Just run them. You don't need to understand every line of code. |
| **⚙️ Try it** | One in Step 5: a value in the code that **you change and re-run**. |
| **💬 Discussion** | At the very end. Answer using the numbers you found along the way. |

If something breaks, first check that you ran **every cell above it**.

**To submit:** answer the Discussion, click **Runtime → Run all**, then **File → Print → Save as PDF**
and upload the PDF to Canvas.

---

## Step 1 — Load the data

This cell loads the dataset straight from the course GitHub page. `scipy.stats` contains the t-distribution and the t-test, so we never have to read a printed t-table — but everything it does, you will also do by hand.

The cell prints each group's mean and the difference between them — the gap the rest of the notebook has to judge.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA = "https://raw.githubusercontent.com/DrGPCR/NEUR201/main/Unit_3/data/"

# Load the table, falling back to a hand-uploaded copy if the download fails
FILENAME = "dFF_per_slice.csv"
try:
    data = pd.read_csv(DATA + FILENAME)
except Exception:
    if not os.path.exists(FILENAME):
        raise SystemExit(
            "\n*** Could not download " + FILENAME + ". ***\n"
            "Click the folder icon in the left sidebar, upload " + FILENAME + ",\n"
            "then run this cell again.")
    data = pd.read_csv(FILENAME)

# Pull the two groups out into their own arrays — we use them in every step below
control = data.loc[data["Group"] == "CONTROL", "dFF_percent"].to_numpy()
drug    = data.loc[data["Group"] == "DRUG",    "dFF_percent"].to_numpy()

print("CONTROL slices:", len(control), "  DRUG slices:", len(drug))
print("CONTROL mean: %5.2f %%" % control.mean())
print("DRUG mean:    %5.2f %%" % drug.mean())
print("Difference (DRUG - CONTROL): %.2f percentage points" % (drug.mean() - control.mean()))

## Step 2 — Standard error of each sample

Whenever you measure a sample, its mean $\bar{X}$ will not exactly equal the population mean $\mu$.
The gap is **random sampling error** — not a mistake, just the consequence of measuring a few slices
instead of all of them. The **standard error** measures how much a sample mean typically wobbles
around $\mu$, and you can estimate it from the one sample you have:

**Step 1 of an independent-samples t-test: find the standard error of each sample separately.**

$$ SE = \frac{s}{\sqrt{n}} $$

where $s$ is the sample standard deviation and $n$ is the number of **slices** in that group.

| | What it measures |
|---|---|
| **Standard deviation** ($s$) | how far **individual slices** sit from their mean — the spread of the **data** |
| **Standard error** ($SE$) | how far **sample means** sit from $\mu$ — the uncertainty of the **estimate** |

The cell below computes $s$ and $SE$ for each group, one line at a time.

In [ ]:
n_control = len(control)
n_drug    = len(drug)

s_control = control.std(ddof=1)     # sample standard deviation (divides by n - 1)
s_drug    = drug.std(ddof=1)

se_control = s_control / np.sqrt(n_control)     # SE = s / sqrt(n)
se_drug    = s_drug    / np.sqrt(n_drug)

print("CONTROL:  s = %.2f,  n = %d   ->   SE = %.2f / sqrt(%d) = %.2f" % (s_control, n_control, s_control, n_control, se_control))
print("DRUG:     s = %.2f,  n = %d   ->   SE = %.2f / sqrt(%d) = %.2f" % (s_drug, n_drug, s_drug, n_drug, se_drug))

## Step 3 — Combined standard error and the t-score

**Step 2: combine the two standard errors.** Both group means wobble, so the standard error of the
**difference** between them is

$$ SE_{\bar{X}_1 - \bar{X}_2} = \sqrt{SE_{\bar{X}_1}^2 + SE_{\bar{X}_2}^2} $$

**Step 3: the t-score.** Count the difference between the means in units of that standard error:

$$ t = \frac{\bar{X}_{\text{DRUG}} - \bar{X}_{\text{CONTROL}}}{SE_{\bar{X}_1 - \bar{X}_2}} $$

Read $t$ as a question: *how many standard errors apart did the two means land?* It is the same idea as
the one-sample t-score, with the difference between **two sample means** on top.

The cell below does each calculation by hand, so every number is visible.

In [ ]:
# Combined standard error: both means wobble, so the squared errors add
se_difference = np.sqrt(se_control**2 + se_drug**2)

# The t-score: the difference between the means, counted in standard errors
difference = drug.mean() - control.mean()
t_score    = difference / se_difference

print("SE of the difference = sqrt(%.2f^2 + %.2f^2) = %.2f" % (se_control, se_drug, se_difference))
print()
print("Difference in means  = %.2f - (%.2f) = %.2f percentage points" % (drug.mean(), control.mean(), difference))
print("t = %.2f / %.2f = %.2f" % (difference, se_difference, t_score))
print()
print("The DRUG mean sits %.2f standard errors %s the CONTROL mean." % (abs(t_score), "above" if t_score > 0 else "below"))

## Step 4 — Degrees of freedom and the p-value

**Step 4: degrees of freedom.** We estimated **two** means, so we lose two degrees of freedom:

$$ df = n_1 + n_2 - 2 $$

**Step 5: the p-value.** The p-value is the probability of getting a t **at least this far from 0, in either
direction, if $H_0$ were true**, i.e. the area in **both tails** beyond $\pm t$. (You can always take the
**absolute value** of $t$ when finding p: the two tails are mirror images.)

Read that sentence twice, because almost everyone gets it backwards: p is the probability of **the data,
given $H_0$**, never the probability that $H_0$ is true.

Python's `scipy.stats` looks up the exact p-value for us. In class you used the printed t-table, which
gives a **range**; both answer the same question. The **critical t** is the t you would need to reach to
have $p = \alpha$ exactly. (We set $\alpha = 0.05$ here.)

In [ ]:
df_ours = n_control + n_drug - 2

p_value = 2 * stats.t.sf(abs(t_score), df_ours)     # area in BOTH tails beyond our t (two-tailed)

alpha  = 0.05                                       # the cut-off, set before the test
t_crit = stats.t.ppf(1 - alpha / 2, df_ours)        # the t you would need to reach p = alpha

# Check against scipy's built-in independent-samples t-test
t_scipy, p_scipy = stats.ttest_ind(drug, control)

print("df = %d + %d - 2 = %d" % (n_control, n_drug, df_ours))
print("t  = %.2f      (|t| = %.2f)" % (t_score, abs(t_score)))
print("p-value (two-tailed) = %.4f" % p_value)
print()
print("alpha = %g   ->   critical t = +/-%.2f" % (alpha, t_crit))
print()
print("Check with scipy:  t = %.2f,  p = %.4f" % (t_scipy, p_scipy))

## Step 5 — Present the result on the t-distribution, then decide

First, the picture. The curve shows how t would behave **if $H_0$ were true**. The **red** area (both
tails beyond $\pm t$) is the **p-value**. The **grey** area is the **rejection region** for your $\alpha$:
the part of the curve beyond the critical t.

> **⚙️ Try it — change alpha.** Run at `alpha = 0.05`, then `0.01`, then `0.001`, and note what
> happens to the grey region and where your t lands. Then set it back to **0.05**.

Then decide. Here is the hypothesis-testing framework from the slides:

Every t-test chooses between two explanations for the gap between two sample means:

| | Claim | Result if true |
|---|---|---|
| **Null hypothesis, $H_0$** | The gap is just **random sampling error** (chance). | statistically **insignificant** |
| **Alternative hypothesis, $H_a$** | The gap **cannot** be explained by chance alone. | statistically **significant** |

A **two-tailed** $H_a$ says the means are *different* without predicting which is larger. For two groups
with population means $\mu_1$ and $\mu_2$ it takes this form:

$$ H_0: \mu_1 = \mu_2 \qquad\qquad H_a: \mu_1 \neq \mu_2 $$

A **one-tailed** $H_a$ predicts a direction (larger *or* smaller) and must be chosen **before** seeing the
data. We use **two-tailed** tests throughout.

**The decision rule.** The cut-off p-value is the **alpha level ($\alpha$)**, conventionally **0.05**:

| If | Decision | Wording |
|---|---|---|
| $p < \alpha$ | **reject $H_0$** | the difference is **statistically significant** |
| $p \geq \alpha$ | **fail to reject $H_0$** | chance cannot be ruled out: **not** significant |

In [ ]:
alpha = 0.05        # ⚙️ Try it — 0.05, then 0.01, then 0.001. Set back to 0.05.

t_crit = stats.t.ppf(1 - alpha / 2, df_ours)        # recompute the critical t for this alpha

lim = max(5, abs(t_score) + 1)
x = np.linspace(-lim, lim, 800)
y = stats.t.pdf(x, df_ours)

plt.figure(figsize=(8, 4.2))
plt.plot(x, y, color="black")
plt.fill_between(x, y, where=np.abs(x) >= t_crit, color="lightgray",
                 label="rejection region (alpha = %g)" % alpha)
plt.fill_between(x, y, where=np.abs(x) >= abs(t_score), color="tomato", alpha=0.85,
                 label="p-value = %.4f (both tails)" % p_value)
plt.axvline(t_score,  color="darkred", lw=2, label="our t = %.2f" % t_score)
plt.axvline(-t_score, color="darkred", lw=1, ls=":")          # the mirror-image tail
for c in (-t_crit, t_crit):
    plt.axvline(c, color="gray", ls="--", lw=1)
plt.xlabel("t")
plt.ylabel("Probability density")
plt.title("t-distribution if H0 were true (df = %d)" % df_ours)
plt.legend(fontsize=8, loc="upper left")
plt.show()

print("t = %.2f,  df = %d,  p = %.4f,  alpha = %g" % (t_score, df_ours, p_value, alpha))

## Putting it all together

| Step | Formula | What it tells you |
|---|---|---|
| 2 | $SE = \dfrac{s}{\sqrt{n}}$ (for each sample) | how much **one group's mean** wobbles by chance |
| 3 | $SE_{\bar{X}_1 - \bar{X}_2} = \sqrt{SE_{\bar{X}_1}^2 + SE_{\bar{X}_2}^2}$ | how much the **difference** between two means wobbles |
| 3 | $t = \dfrac{\bar{X}_{\text{DRUG}} - \bar{X}_{\text{CONTROL}}}{SE_{\bar{X}_1 - \bar{X}_2}}$ | the difference, counted in standard errors |
| 4 | $df = n_1 + n_2 - 2$ | which t-distribution to use |
| 4 | $p$ = area in both tails beyond $\pm t$ | how surprising that t would be **if $H_0$ were true** |
| 5 | $p < \alpha$ → reject $H_0$ | the decision |

Four things to carry forward:

- **$n$ is under a square root.** Quadrupling your sample halves your standard error, and no more.
- The p-value is the probability of **the data given $H_0$**, never the probability that $H_0$ is true.
- **Fail to reject** is not **accept**: a small experiment fails to reject almost everything.
- **Significant is not large.** Neither the p-value nor the stars tell you whether a 4-percentage-point
  change in calcium activity *matters* biologically — that judgement is still yours.

---

### 💬 Discussion

Keep each answer **short** (a number and one or two sentences is often enough). Use the default ⚙️ values
unless a question says otherwise.

1. **t-score (Step 3).** State the **two-tailed** $H_0$ and $H_a$ for this experiment as plain sentences
   about slices and calcium (you may add the symbols $\mu_{\text{DRUG}}$ and $\mu_{\text{CONTROL}}$). Then report
   the SE of the difference and the **t-score**, and say in one sentence what your t is counting.

2. **p-value (Step 4).** Report the **degrees of freedom** and the **p-value**. Finish this sentence:
   *"If the null hypothesis were true…"*

3. **Decision at $\alpha = 0.05$ (Step 5).** Compare your p-value with $\alpha = 0.05$. Is the difference
   statistically significant? Say whether you **reject** or **fail to reject** $H_0$, and why.

4. **Try two different alphas (Step 5).** Use the Try it to set `alpha` to **0.01** and then **0.001**,
   re-running each time. For each one, do you **reject** or **fail to reject** $H_0$? Did the p-value
   change? Then explain, in one or two sentences, what your results mean **biologically** for 4-AP's
   effect on calcium activity.

*(Before you submit: set `alpha = 0.05` in Step 5, then run all.)*

**Your answer:** *(double-click here to type)*

>
>
>
>
>

---
### Nice work — you're done!

Click **Runtime → Run all**, then **File → Print → Save as PDF**, and upload the PDF to Canvas.

This cell is empty on purpose. Do not delete.